# Revisão de Classificações Pendentes (> 5% Não Classificado)

Este notebook foi criado para facilitar a identificação e categorização de dados dimensionais que permanecem como `Não classificado`, `Não informado`, `Outros / Revisar` etc. na camada de BI.

## Passo a Passo
1. Rode a primeira célula de código para listar quais tabelas e colunas têm **mais de 5% de dados não classificados**.
2. Rode a segunda célula para gerar automaticamente **templates de dicionários (`.csv`)** na pasta `data/04_review/dicionarios_manuais/` contendo os valores únicos dessas colunas que precisam ser mapeados.
3. Abra os arquivos `.csv` gerados pelo Excel ou bloco de notas, preencha as colunas de normalização com suas respectivas categorias e salve-os.
4. Ao executar o pipeline do BI novamente, o `src.bi.dimensions` fará o merge com os seus dicionários (lembre-se de registrar a nova lógica no `dimensions.py` caso o dicionário seja inteiramente novo).

In [ ]:
import pandas as pd
import glob
import os

base_dir = r'../data/04_bi_ready/dimensoes'
csv_files = glob.glob(os.path.join(base_dir, '*.csv'))

# Termos que indicam necessidade de categorização
unclassified_terms = [
    'não classificado', 'nao classificado', 
    'não informada', 'nao informada', 
    'não informado', 'nao informado', 
    'outros / revisar', 'revisar manualmente',
    'não classificada', 'nao classificada'
]

print("=== Relatório de Não Classificados (> 5%) ===\n")

columns_to_review = []

for file in csv_files:
    df = pd.read_csv(file, encoding='utf-8-sig')
    file_name = os.path.basename(file)
    total_rows = len(df)
    
    for col in df.columns:
        if df[col].dtype == object:
            series = df[col].astype(str).str.lower().str.strip()
            # Conta quantos valores batem com a lista de não classificados
            count = series.isin(unclassified_terms).sum()
            
            if count > 0:
                pct = (count / total_rows) * 100
                if pct >= 5.0:
                    print(f"[{file_name}] Coluna: '{col}' | Não Classificados: {count}/{total_rows} ({pct:.2f}%)")
                    columns_to_review.append({
                        'file': file, 
                        'file_name': file_name,
                        'column': col,
                        'pct': pct
                    })


### Geração dos Templates para Dicionário
A célula abaixo criará arquivos `.csv` na pasta de dicionários manuais com os valores únicos de cada tabela que ultrapassou o limite de 5%.

In [ ]:
review_dir = r'../data/04_review/dicionarios_manuais'
os.makedirs(review_dir, exist_ok=True)

for item in columns_to_review:
    df = pd.read_csv(item['file'], encoding='utf-8-sig')
    col_name = item['column']
    file_name = item['file_name'].replace('.csv', '')
    
    # Extrair valores únicos, removendo os próprios "Não classificados"
    unique_vals = df[col_name].dropna().unique()
    unique_vals = [v for v in unique_vals if str(v).lower().strip() not in unclassified_terms]
    
    # Se houver outra coluna que geralmente serve de chave de origem (ex: 'instituicao' quando a coluna alvo é 'tipo_instituicao')
    # Vamos tentar pegar o valor original em vez de apenas o valor alvo que está como 'não classificado'
    # Para facilitar, exportamos a tabela inteira reduzida a valores onde a coluna alvo é nula ou não classificada
    df_pendente = df[df[col_name].astype(str).str.lower().str.strip().isin(unclassified_terms)].copy()
    
    # Drop chaves e colunas técnicas
    cols_to_drop = [c for c in df_pendente.columns if c.startswith('sk_') or c.startswith('id_') or c.startswith('qtd_')]
    df_pendente = df_pendente.drop(columns=cols_to_drop)
    
    # Deixa apenas valores unicos baseados em todas as colunas que restaram
    df_pendente = df_pendente.drop_duplicates()
    
    out_file = os.path.join(review_dir, f"revisao_pendente_{file_name}_{col_name}.csv")
    df_pendente.to_csv(out_file, index=False, encoding='utf-8-sig')
    print(f"Template criado: {out_file} ({len(df_pendente)} linhas pendentes para revisar)")

print("\nProcesso concluído!")
